In [ ]:
!pip install -q transformers accelerate gradio PyPDF2 python-docx scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.0 MB/s eta 0:00:00


In [ ]:
# =============================
# Install Required Libraries
# =============================
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import PyPDF2
import docx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =============================
# Load Model
# =============================
model_name = "ibm-granite/granite-3.2-2b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# =============================
# Extract Resume Text
# =============================
def extract_text(file):
    text = ""
    try:
        if file.name.endswith(".pdf"):
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() or ""
        elif file.name.endswith(".docx"):
            doc = docx.Document(file)
            for para in doc.paragraphs:
                text += para.text + "\n"
    except:
        return "Error reading file."

    return text[:2000]


# =============================
# Compute Similarity Score
# =============================
def compute_similarity(resume_text, job_description):
    vectorizer = TfidfVectorizer(stop_words="english")
    vectors = vectorizer.fit_transform([resume_text, job_description])
    score = cosine_similarity(vectors[0], vectors[1])[0][0]
    return round(score * 100, 2)


# =============================
# Generate SHORT Response
# =============================
def generate_response(prompt):

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,   # 🔥 Reduced for short answer
            temperature=0.4,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).replace(prompt, "").strip()


# =============================
# Analyze Multiple Resumes (Short Version)
# =============================
def analyze_multiple_resumes(files, job_description):

    if not files:
        return "Please upload at least one resume."

    final_output = ""

    for i, file in enumerate(files):
        resume_text = extract_text(file)

        if resume_text == "Error reading file.":
            continue

        similarity_score = compute_similarity(resume_text, job_description)

        prompt = f"""
You are an ATS Resume Analyzer.

Give SHORT and concise output.
Limit each section to 1–2 lines only.

Format:

ATS Score:

Similarity Score: {similarity_score}%

Resume:
{resume_text}

Job Description:
{job_description}
"""

        result = generate_response(prompt)

        final_output += f"\n\n📄 Resume {i+1}\n"
        final_output += result + "\n"

    return final_output


# =============================
# Clean Dark UI
# =============================
dark_css = """
html, body {
    background-color: #111111 !important;
}

.gradio-container {
    background-color: #111111 !important;
    max-width: 100% !important;
}

h1, h2, h3, p, label {
    color: #e0e0e0 !important;
}

textarea, input {
    background-color: #1c1c1c !important;
    color: #e0e0e0 !important;
    border: 1px solid #2a2a2a !important;
    border-radius: 12px !important;
}

button {
    background-color: #2b2b2b !important;
    color: #e0e0e0 !important;
    border-radius: 12px !important;
    border: 1px solid #3a3a3a !important;
}

button:hover {
    background-color: #3a3a3a !important;
}

footer {
    display: none !important;
}
"""

# =============================
# UI
# =============================
with gr.Blocks(css=dark_css) as app:

    gr.Markdown("# 💬 AI Resume Analyzer")
    gr.Markdown("Upload multiple resumes for short ATS analysis.")

    resume_input = gr.File(label="Upload Resumes (PDF/DOCX)", file_count="multiple")
    job_desc_input = gr.Textbox(label="Job Description", lines=6)

    analyze_btn = gr.Button("Analyze")

    output_box = gr.Textbox(label="Result", lines=25)

    analyze_btn.click(
        analyze_multiple_resumes,
        inputs=[resume_input, job_desc_input],
        outputs=output_box
    )

app.launch(share=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

/tmp/ipykernel_326/1671570207.py:168: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=dark_css) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b121d22ef4959ffe9e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
